# Image Classification: From Pixels to Predictions

A complete end-to-end image classification pipeline using `sklearn.datasets.load_digits` (8×8 images, 10 classes) as a proxy for real-world datasets like MNIST or CIFAR.

We compare three broad approaches:
1. **Classic ML** on flattened pixels (Logistic Regression, SVM, Random Forest)
2. **Dimensionality Reduction + Classifier** (PCA + Logistic Regression)
3. **Simple Neural Network** (TensorFlow/Keras feedforward network)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

---
## 1. Problem Statement

**Goal**: Classify handwritten digits (0–9) from 8×8 grayscale images.

- **Dataset**: 1 797 samples, each an 8×8 image (64 pixels per sample)
- **Classes**: 10 (digits 0 through 9)
- **Challenge**: High-dimensional pixel space makes classification non-trivial

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
images = digits.images  # original 8x8 arrays

print(f"Data shape (flattened): {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Image size: {images.shape[1]}x{images.shape[2]}")

---
## 2. Data Exploration

### 2.1 Visualise Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, label in zip(axes.ravel(), images, y):
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Digit: {label}')
    ax.axis('off')
fig.suptitle('Sample Handwritten Digits', fontsize=14)
plt.tight_layout()
plt.savefig('sample_digits.png', bbox_inches='tight')
plt.show()

### 2.2 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(x=y, ax=ax, palette='viridis')
ax.set_title('Class Distribution')
ax.set_xlabel('Digit')
ax.set_ylabel('Count')
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

---
## 3. Data Preprocessing

- Normalise pixel values to [0, 1] range
- The `load_digits` data originally ranges from 0 to 16 (greyscale)
- We keep both the **flattened** representation (for classic ML) and 2D structure (for visualisation)
- Train/test split: 80 % / 20 %

In [ ]:
X = X / 16.0  # normalise to [0, 1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

---
## 4. Approach 1 — Classic ML on Flattened Pixels

We benchmark three classifiers:
- **Logistic Regression** (multinomial, max_iter=5000)
- **SVM** with RBF kernel
- **Random Forest** (100 trees)

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(multi_class='multinomial', max_iter=5000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', gamma='scale', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
trained_models = {}

for name, clf in classifiers.items():
    start = time.time()
    clf.fit(X_train, y_train)
    elapsed = time.time() - start
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': acc, 'Train Time (s)': elapsed})
    trained_models[name] = clf
    print(f"{name:25s}  Accuracy: {acc:.4f}  |  Train time: {elapsed:.3f}s")

results_df = pd.DataFrame(results)
results_df

### 4.1 Performance Comparison (Classic ML)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results_df['Model'], results_df['Accuracy'], color=['#3498db', '#e74c3c', '#2ecc71'])
ax.set_ylim(0.9, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Classic ML — Accuracy on Test Set')
for bar, acc in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003, f'{acc:.4f}',
            ha='center', va='bottom', fontsize=10)
plt.savefig('classic_ml_accuracy.png', bbox_inches='tight')
plt.show()

---
## 5. Approach 2 — PCA + Classifier

Principal Component Analysis reduces the 64-dimensional pixel space to a low-dimensional representation. This helps visualise the data and can speed up training.

### 5.1 PCA 2-D Projection

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.7, s=30)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')
ax.set_title('Digits Dataset — PCA 2-D Projection')
fig.colorbar(scatter, ax=ax, ticks=range(10), label='Digit')
plt.savefig('pca_2d_projection.png', bbox_inches='tight')
plt.show()

### 5.2 Accuracy vs. Number of PCA Components

In [ ]:
components = range(1, 65, 2)
pca_scores = []

for n in components:
    pca = PCA(n_components=n, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    clf = LogisticRegression(multi_class='multinomial', max_iter=5000, random_state=42)
    clf.fit(X_train_pca, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_pca))
    pca_scores.append(acc)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(components, pca_scores, marker='o', linestyle='-', color='#8e44ad')
ax.axhline(y=results_df.loc[results_df['Model'] == 'Logistic Regression', 'Accuracy'].values[0],
           color='gray', linestyle='--', label='Full-dim LR baseline')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs. Number of PCA Components')
ax.legend()
plt.savefig('accuracy_vs_pca.png', bbox_inches='tight')
plt.show()

print(f"Best PCA accuracy: {max(pca_scores):.4f} with {components[np.argmax(pca_scores)]} components")

### 5.3 Classify with PCA-Reduced Features

We pick the optimal number of components (or a reasonable default like 30) and run the same classifiers.

In [ ]:
n_components = 30
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

pca_results = []
for name, clf in classifiers.items():
    start = time.time()
    clf.fit(X_train_pca, y_train)
    elapsed = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test_pca))
    pca_results.append({'Model': name + ' (PCA)', 'Accuracy': acc, 'Train Time (s)': elapsed})
    print(f"{name + ' (PCA)':30s}  Accuracy: {acc:.4f}  |  Train time: {elapsed:.3f}s")

pca_results_df = pd.DataFrame(pca_results)
pca_results_df

---
## 6. Approach 3 — Simple Neural Network (TensorFlow / Keras)

We build a small feedforward network to demonstrate a deep-learning workflow:
1. `Sequential` model with `Dense` layers
2. ReLU activations for hidden layers, softmax for output
3. Categorical cross-entropy loss
4. Train for 50 epochs with a validation split

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

model = Sequential([
    Dense(128, activation='relu', input_shape=(64,)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
start = time.time()
history = model.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)
nn_train_time = time.time() - start
print(f"\n Neural network training completed in {nn_train_time:.2f}s")

### 6.1 Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train loss')
ax1.plot(history.history['val_loss'], label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train acc')
ax2.plot(history.history['val_accuracy'], label='Val acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curves')
ax2.legend()

plt.savefig('learning_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
nn_y_pred = model.predict(X_test)
nn_y_pred_classes = np.argmax(nn_y_pred, axis=1)
nn_acc = accuracy_score(y_test, nn_y_pred_classes)
print(f"Neural Network Test Accuracy: {nn_acc:.4f}")

nn_result = {'Model': 'Neural Network', 'Accuracy': nn_acc, 'Train Time (s)': nn_train_time}

---
## 7. Confusion Matrix Analysis

Which pairs of digits does the model confuse most often? We use the Neural Network predictions as our reference model.

In [ ]:
cm = confusion_matrix(y_test, nn_y_pred_classes)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — Neural Network')
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

# Identify the most confused pair (largest off-diagonal)
np.fill_diagonal(cm, 0)
most_confused = np.unravel_index(np.argmax(cm), cm.shape)
print(f"Most confused pair: true={most_confused[0]}, predicted={most_confused[1]} "
      f"({cm[most_confused]} misclassifications)")

---
## 8. Visualising Misclassified Examples

Let's look at the images that the Neural Network got wrong.

In [ ]:
misclassified = np.where(y_test != nn_y_pred_classes)[0]
print(f"Total misclassified: {len(misclassified)} out of {len(y_test)}")

n_show = min(16, len(misclassified))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, idx in enumerate(misclassified[:n_show]):
    ax = axes.ravel()[i]
    ax.imshow(X_test[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'True: {y_test[idx]}\nPred: {nn_y_pred_classes[idx]}', fontsize=9)
    ax.axis('off')
if n_show < 16:
    for i in range(n_show, 16):
        axes.ravel()[i].axis('off')
fig.suptitle('Misclassified Examples', fontsize=14)
plt.tight_layout()
plt.savefig('misclassified.png', bbox_inches='tight')
plt.show()

---
## 9. Final Comparison — All Approaches

Merge all results into a single DataFrame and produce a bar chart comparing both accuracy and training time.

In [ ]:
all_results = pd.concat([results_df, pca_results_df], ignore_index=True)
all_results = pd.concat([all_results, pd.DataFrame([nn_result])], ignore_index=True)
all_results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(np.linspace(0, 1, len(all_results)))

bars1 = ax1.barh(all_results['Model'], all_results['Accuracy'], color=colors)
ax1.set_xlabel('Accuracy')
ax1.set_title('Model Comparison — Accuracy')
for bar, acc in zip(bars1, all_results['Accuracy']):
    ax1.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
             f'{acc:.4f}', va='center', fontsize=9)
ax1.set_xlim(0.85, 1.0)

bars2 = ax2.barh(all_results['Model'], all_results['Train Time (s)'], color=colors)
ax2.set_xlabel('Train Time (s)')
ax2.set_title('Model Comparison — Training Time')
for bar, t in zip(bars2, all_results['Train Time (s)']):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
             f'{t:.2f}s', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

---
## 10. Summary

| Approach | Best Accuracy | Training Speed | Notes |
|----------|--------------|----------------|-------|
| **Classic ML** (SVM / RF) | ~0.97–0.98 | Very fast (≈0.1s) | Strong baseline, no feature engineering needed |
| **PCA + Classifier** | ~0.97 (with 30 components) | Fast | Useful for visualisation & when interpretability matters |
| **Neural Network** | ~0.97–0.98 | Moderate (≈10–20s) | Scales to larger datasets; benefits from more data |

Key takeaways:
- Even simple pixel-level classifiers achieve high accuracy on this small dataset.
- PCA helps visualise high-dimensional image data in 2D.
- Neural networks become increasingly advantageous as dataset size grows.
- Misclassification analysis reveals which digit pairs are intrinsically hard to separate (e.g., 8↔9, 3↔5).

---
*End-to-end image classification project — built with scikit-learn, TensorFlow/Keras, and Matplotlib.*